# Resultados UNAM — Distribución de aciertos por carrera y año

Comparación histórica del examen de ingreso a licenciatura de la UNAM a partir de los datos publicados por la DGAE (`dgae.unam.mx`).

Cada fila del dataset representa un aspirante (folio), con su número de aciertos y si fue acreditado. Los datos se obtuvieron directamente de las páginas de resultados oficiales — ver `src/fetch_and_parse.py` para el proceso de extracción.

**Contenido:**
1. Carga y limpieza de datos
2. Resumen por carrera y año
3. Distribución de aciertos — comparación interactiva entre años
4. Evolución del acierto mínimo de corte por carrera


## 1. Carga y limpieza de datos

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

pd.set_option("display.max_columns", None)

In [2]:
DATA_PATH = "../data/processed/aciertos_unam_2021_2026.csv"

df = pd.read_csv(DATA_PATH)
print(f"Filas totales: {len(df):,}")
df.head()

Filas totales: 1,027,716


,folio,aciertos,acreditado,año,area,carrera,plantel
0,1,71.0,NaN,2021,1,ACTUARIA,FACULTAD DE CIENCIAS
1,30,93.0,NaN,2021,1,ACTUARIA,FACULTAD DE CIENCIAS
2,56,49.0,NaN,2021,1,ACTUARIA,FACULTAD DE CIENCIAS
3,133,90.0,NaN,2021,1,ACTUARIA,FACULTAD DE CIENCIAS
4,148,62.0,NaN,2021,1,ACTUARIA,FACULTAD DE CIENCIAS


In [3]:
# Nos quedamos solo con quienes presentaron examen (aciertos no nulo)
# Los folios sin aciertos son aspirantes registrados que no presentaron (N) o fueron cancelados (C)
presentaron = df[df["aciertos"].notna()].copy()
presentaron["aciertos"] = presentaron["aciertos"].astype(int)
presentaron["año"] = presentaron["año"].astype(int)

print(f"Aspirantes registrados: {len(df):,}")
print(f"Presentaron examen:     {len(presentaron):,}")
print(f"\nCarreras distintas: {presentaron['carrera'].nunique()}")
print(f"Años cubiertos: {sorted(presentaron['año'].unique())}")

Aspirantes registrados: 1,027,716
Presentaron examen:     874,022

Carreras distintas: 116
Años cubiertos: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## 2. Resumen por carrera y año

In [4]:
resumen = (
    presentaron
    .groupby(["carrera", "año"])
    .agg(
        aspirantes=("folio", "count"),
        aciertos_promedio=("aciertos", "mean"),
        aciertos_mínimo=("aciertos", "min"),
        aciertos_máximo=("aciertos", "max"),
        seleccionados=("acreditado", lambda s: (s == "S").sum()),
    )
    .round(1)
    .reset_index()
    .sort_values(["carrera", "año"])
)
resumen

,carrera,año,aspirantes,aciertos_promedio,aciertos_mínimo,aciertos_máximo,seleccionados
0,ACTUARIA,2021,1463,67.9,20,120,103
1,ACTUARIA,2022,1312,65.6,23,118,105
2,ACTUARIA,2023,1332,64.7,23,116,112
3,ACTUARIA,2024,1539,67.2,26,117,108
4,ACTUARIA,2025,2005,67.1,24,118,105
...,...,...,...,...,...,...,...
662,URBANISMO,2022,104,58.9,22,115,24
663,URBANISMO,2023,104,60.1,29,102,21
664,URBANISMO,2024,101,55.9,26,104,24
665,URBANISMO,2025,180,57.7,22,115,27


## 3. Distribución de aciertos — comparación interactiva entre años

Selecciona una carrera en el menú desplegable para ver cómo cambió la distribución de aciertos año con año. La línea vertical marca el acierto mínimo de corte de cada año (nota: el corte se calcula sobre el conjunto agregado de planteles de esa carrera en este dataset).

In [ ]:
def graficar_distribucion(selección):
    area_sel, carrera_sel = selección
    datos = presentaron[(presentaron["area"] == area_sel) & (presentaron["carrera"] == carrera_sel)]

    fig = px.histogram(
        datos, x="aciertos", color="año", barmode="overlay", opacity=0.55, nbins=40,
        histnorm="percent", color_discrete_sequence=px.colors.qualitative.Set2,
        labels={"aciertos": "Aciertos", "percent": "% de sustentantes", "año": "Año"},
        title=f"Distribución de aciertos — {carrera_sel} (Área {area_sel})",
    )

    for año, grupo in datos.groupby("año"):
        seleccionados = grupo[grupo["acreditado"] == "S"]
        if len(seleccionados) > 0:
            corte = seleccionados["aciertos"].min()
            fig.add_vline(
                x=corte, line_dash="dot", line_width=1,
                annotation_text=f"{año}: corte {corte}", annotation_position="top", opacity=0.6,
            )

    fig.update_layout(template="plotly_white", legend_title_text="Año", height=520, bargap=0.05)
    fig.show()


opciones = sorted(presentaron[["area", "carrera"]].drop_duplicates().itertuples(index=False, name=None))
etiquetas = {f"Área {a} — {c}": (a, c) for a, c in opciones}

selector = widgets.Dropdown(
    options=etiquetas, 
    description="Carrera:",
    style={"description_width": "initial"},
)

widgets.interact(graficar_distribucion, selección=selector);

interactive(children=(Dropdown(description='Carrera:', options={'Área 1 — ACTUARIA': (1, 'ACTUARIA'), 'Área 1 …

---
### Notas metodológicas

- Los datos agregan todos los planteles de cada carrera (ej. Ingeniería Civil en Facultad de Ingeniería + FES Acatlán + FES Aragón se cuentan juntos). Para un análisis por plantel, filtra sobre la columna `plantel` antes de graficar.
- El histograma está normalizado (`histnorm="percent"`) porque el número de sustentantes cambia significativamente entre años y carreras — comparar conteos absolutos sería engañoso.
- Fuente: Dirección General de Administración Escolar (DGAE), UNAM — `dgae.unam.mx`. Ver `LICENSE-DATA` para términos de reuso de este dataset.